In [1]:
import pandas as pd

# 1. ส่วนกำหนดข้อมูล (สามารถ เพิ่ม/ลบ/แก้ไข รายการสินค้าตรงนี้ได้เลย)
# ข้อมูลจำลองตามตารางที่ 5.2 ในภาพ
products_data = [
    {"product": "A", "sales": 1000000, "price": 2.50},
    {"product": "B", "sales": 250000, "price": 0.55},
    {"product": "C", "sales": 150000, "price": 6.50},
    {"product": "D", "sales": 300000, "price": 1.00},
    {"product": "E", "sales": 100000, "price": 1.50},
    {"product": "F", "sales": 700000, "price": 1.43},
    {"product": "G", "sales": 500000, "price": 9.00},
    {"product": "H", "sales": 15000, "price": 4.98},
    {"product": "J", "sales": 1000000, "price": 0.75},
    {"product": "K", "sales": 600000, "price": 1.62},
    {"product": "L", "sales": 25000, "price": 2.50},
    {"product": "M", "sales": 4200, "price": 15.00},
    {"product": "N", "sales": 1000000, "price": 5.00},
    {"product": "O", "sales": 2850000, "price": 10.00},
    {"product": "P", "sales": 10000, "price": 0.83},
    {"product": "Q", "sales": 355000, "price": 0.99},
    {"product": "R", "sales": 40000, "price": 1.37},
    {"product": "S", "sales": 393000, "price": 1.85},
    {"product": "T", "sales": 250000, "price": 4.12}, 
]

# แปลงข้อมูลเป็น DataFrame
df = pd.DataFrame(products_data)

# 2. ขั้นตอนการคำนวณ
# 2.1 คำนวณมูลค่ารวม (ยอดขาย x ราคา)
df['total_value'] = df['sales'] * df['price']

# 2.2 เรียงลำดับจากมูลค่ามากไปน้อย (สำคัญมากสำหรับ ABC Analysis)
df = df.sort_values(by='total_value', ascending=False).reset_index(drop=True)

# 2.3 คำนวณมูลค่าสะสม (Cumulative Value)
df['accumulated_value'] = df['total_value'].cumsum()

# 2.4 คำนวณ % สะสม
total_sum = df['total_value'].sum()
df['percent_accumulate'] = (df['accumulated_value'] / total_sum) * 100

# 3. การจัดกลุ่ม ABC (ตามเกณฑ์ 80-20 หรือตามภาพที่ 5.3)
# เกณฑ์: A <= 80%, B <= 95%, C > 95% (ปรับแก้ตัวเลขได้ตามต้องการ)
def assign_group(pct):
    # หมายเหตุ: ใช้ 80.1 เพื่อให้ครอบคลุมกรณีที่ตัวเลขเกิน 80 มานิดหน่อยเหมือนในตัวอย่าง (เช่น 80.05)
    if pct <= 80.1: 
        return 'A'
    elif pct <= 95.5:
        return 'B'
    else:
        return 'C'

df['group'] = df['percent_accumulate'].apply(assign_group)

# 4. จัดรูปแบบการแสดงผลให้สวยงามเหมือนตารางที่ 5.3
# เปลี่ยนชื่อคอลัมน์ภาษาไทย
output_df = df.rename(columns={
    'product': 'ผลิตภัณฑ์',
    'sales': 'ยอดขาย (ชิ้น)',
    'price': 'ราคา (บาท/ชิ้น)',
    'total_value': 'มูลค่ารวม',
    'percent_accumulate': '% สะสม',
    'group': 'กลุ่ม'
})

# จัดรูปแบบตัวเลข (ใส่ลูกน้ำและทศนิยม)
pd.options.display.float_format = '{:,.2f}'.format

# แสดงผลลัพธ์
display(output_df)

# สรุปยอดตามกลุ่ม
print("\n--- สรุปข้อมูลตามกลุ่ม ---")
summary = output_df.groupby('กลุ่ม')['มูลค่ารวม'].agg(['count', 'sum'])
summary['% ของมูลค่าทั้งหมด'] = (summary['sum'] / total_sum) * 100
display(summary)

,ผลิตภัณฑ์,ยอดขาย (ชิ้น),ราคา (บาท/ชิ้น),มูลค่ารวม,accumulated_value,% สะสม,กลุ่ม
0,O,2850000,10.00,"28,500,000.00","28,500,000.00",60.44,A
1,N,1000000,5.00,"5,000,000.00","33,500,000.00",71.04,A
2,G,500000,9.00,"4,500,000.00","38,000,000.00",80.58,B
3,A,1000000,2.50,"2,500,000.00","40,500,000.00",85.88,B
4,T,250000,4.12,"1,030,000.00","41,530,000.00",88.07,B
5,F,700000,1.43,"1,001,000.00","42,531,000.00",90.19,B
6,C,150000,6.50,"975,000.00","43,506,000.00",92.26,B
7,K,600000,1.62,"972,000.00","44,478,000.00",94.32,B
8,J,1000000,0.75,"750,000.00","45,228,000.00",95.91,C
9,S,393000,1.85,"727,050.00","45,955,050.00",97.45,C



--- สรุปข้อมูลตามกลุ่ม ---


,count,sum,% ของมูลค่าทั้งหมด
กลุ่ม,,,
A,2,"33,500,000.00",71.04
B,6,"10,978,000.00",23.28
C,11,"2,679,300.00",5.68


In [2]:
import pandas as pd
import numpy as np

# 1. ส่วนกำหนดข้อมูล (สามารถ เพิ่ม/ลบ/แก้ไข รายการสินค้าตรงนี้ได้เลย)
# ข้อมูลเริ่มต้นคือตัวอย่างที่ 5.2 จากภาพ
inventory_data = [
    {
        "product_name": "ตัวอย่างที่ 5.2",
        "D": 8000,       # ความต้องการ (หน่วย/ปี)
        "P": 700,        # ค่าใช้จ่ายในการสั่งซื้อ (บาท/ครั้ง)
        "C": 1200,       # ต้นทุนผลิตภัณฑ์ (บาท/หน่วย)
        "i_percent": 20, # ดอกเบี้ย (% ต่อปี)
        "W_monthly": 18  # ค่าเช่าพื้นที่เก็บ (บาท/หน่วย/เดือน)
    },
    # คุณสามารถเพิ่มสินค้าอื่น ๆ ต่อท้ายได้ เช่น:
    # {
    #     "product_name": "สินค้า B",
    #     "D": 5000, "P": 500, "C": 800, "i_percent": 15, "W_monthly": 10
    # },
]

# 2. ฟังก์ชันคำนวณ
def calculate_inventory_costs(data_list):
    results = []
    
    for item in data_list:
        # ดึงตัวแปร
        D = item['D']
        P = item['P']
        C = item['C']
        i = item['i_percent'] / 100
        W_mo = item['W_monthly']
        
        # คำนวณค่าใช้จ่ายต่อปี (I และ W)
        # I = ค่าใช้จ่ายเกี่ยวกับอัตราผลตอบแทน (บาท/หน่วย/ปี)
        I = i * C
        # W = ค่าพื้นที่เก็บ (บาท/หน่วย/ปี) = เดือนละ 18 * 12 เดือน
        W = W_mo * 12
        
        # --- กรณีที่ 1: นโยบายแบบสุ่ม (Randomized Storage) ---
        # สูตร Q_opt = sqrt( 2DP / (I + W) )  [จากสมการ 5.9 ในภาพ]
        # ตัวหารคือ I + W เพราะคิดพื้นที่จากค่าเฉลี่ย
        denom_random = I + W
        Q_random = np.sqrt((2 * D * P) / denom_random)
        
        # คำนวณ TC (Total Cost per Unit)
        # TC = C + P/Q + (I+W)Q / 2D [จากสมการ 5.7 ในภาพ]
        holding_cost_random = (denom_random * Q_random) / (2 * D)
        TC_random = C + (P / Q_random) + holding_cost_random

        # --- กรณีที่ 2: นโยบายแบบกำหนดพื้นที่ (Dedicated Storage) ---
        # สูตร Q_opt = sqrt( 2DP / (I + 2W) ) [จากสมการ 5.13 ในภาพ]
        # ตัวหารคือ I + 2W (มาจาก I + W ที่ต้องคูณ 2 เพราะจองพื้นที่สูงสุด Q ไม่ใช่ Q/2)
        denom_dedicated = I + (2 * W)
        Q_dedicated = np.sqrt((2 * D * P) / denom_dedicated)
        
        # คำนวณ TC (Total Cost per Unit)
        # TC = C + P/Q + I(Q/2)/D + W(Q)/D [จากสมการ 5.11 ในภาพ]
        # หมายเหตุ: ในภาพมีการพิมพ์ผิดเล็กน้อยตรงการแทนค่า P/Q (ใช้ 157 แทนที่จะเป็น 130) แต่ผลลัพธ์สุดท้ายถูกต้อง
        interest_cost_dedicated = (I * (Q_dedicated / 2)) / D
        storage_cost_dedicated = (W * Q_dedicated) / D
        TC_dedicated = C + (P / Q_dedicated) + interest_cost_dedicated + storage_cost_dedicated

        # เก็บผลลัพธ์
        results.append({
            "สินค้า": item['product_name'],
            "ความต้องการ (D)": D,
            "I (บาท/ปี)": I,
            "W (บาท/ปี)": W,
            # ผลลัพธ์แบบสุ่ม
            "Q* (แบบสุ่ม)": Q_random,
            "TC/หน่วย (แบบสุ่ม)": TC_random,
            # ผลลัพธ์แบบกำหนดพื้นที่
            "Q* (กำหนดพื้นที่)": Q_dedicated,
            "TC/หน่วย (กำหนดพื้นที่)": TC_dedicated,
            # ส่วนต่าง
            "ผลต่าง TC": TC_dedicated - TC_random
        })
        
    return pd.DataFrame(results)

# 3. ประมวลผลและแสดงผล
df_result = calculate_inventory_costs(inventory_data)

# จัดรูปแบบการแสดงผลตัวเลข
pd.options.display.float_format = '{:,.2f}'.format

# แสดงตาราง
print(f"ผลการคำนวณเปรียบเทียบนโยบายการสั่งซื้อ (อ้างอิงข้อมูล: {inventory_data[0]['product_name']})")
display(df_result.T) # .T เพื่อสลับแกนให้อ่านง่ายขึ้นสำหรับข้อมูลเดียว

ผลการคำนวณเปรียบเทียบนโยบายการสั่งซื้อ (อ้างอิงข้อมูล: ตัวอย่างที่ 5.2)


,0
สินค้า,ตัวอย่างที่ 5.2
ความต้องการ (D),8000
I (บาท/ปี),240.00
W (บาท/ปี),216
Q* (แบบสุ่ม),156.72
TC/หน่วย (แบบสุ่ม),"1,208.93"
Q* (กำหนดพื้นที่),129.10
TC/หน่วย (กำหนดพื้นที่),"1,210.84"
ผลต่าง TC,1.91


In [3]:
import pandas as pd
import numpy as np
import math

# ==========================================
# 1. ส่วนกำหนดข้อมูล (User Input)
# ==========================================
# คุณสามารถ เพิ่ม/ลบ สินค้าใน list นี้ได้ตามต้องการ
inventory_data = [
    {
        "product_name": "สินค้าตัวอย่าง (Ex 5.2 & 5.3)",
        "D": 8000,        # ความต้องการ (หน่วย/ปี)
        "P": 700,         # ค่าใช้จ่ายในการสั่งซื้อ (บาท/ครั้ง)
        "C": 1200,        # ต้นทุนผลิตภัณฑ์ (บาท/หน่วย)
        "i_percent": 20,  # ดอกเบี้ย (% ต่อปี)
        "W_monthly": 18,  # ค่าเช่าพื้นที่เก็บ (บาท/หน่วย/เดือน)
        "L": 2,           # Lead Time (วัน) - สำหรับ Ex 5.3
        "working_days": 300 # จำนวนวันทำงานต่อปี - สำหรับ Ex 5.3
    },
    # ตัวอย่างการเพิ่มสินค้าใหม่ (ลอง Uncomment เพื่อทดสอบได้)
    # {
    #     "product_name": "สินค้า B",
    #     "D": 5000, "P": 500, "C": 800, "i_percent": 15, "W_monthly": 10,
    #     "L": 5, "working_days": 365
    # },
]

# ==========================================
# 2. ฟังก์ชันคำนวณ
# ==========================================
def analyze_inventory(data_list):
    cost_analysis_results = [] # ผลลัพธ์ส่วนที่ 1 (Ex 5.2)
    operational_plan_results = [] # ผลลัพธ์ส่วนที่ 2 (Ex 5.3)

    for item in data_list:
        # --- ดึงตัวแปร ---
        D = item['D']
        P = item['P']
        C = item['C']
        i = item['i_percent'] / 100
        W_mo = item['W_monthly']
        L = item['L']
        days_yr = item['working_days']

        # คำนวณค่าคงที่
        I = i * C              # Interest Charge (บาท/หน่วย/ปี)
        W = W_mo * 12          # Storage Charge (บาท/หน่วย/ปี)

        # -------------------------------------------------------
        # ส่วนที่ 1: การวิเคราะห์ต้นทุนทางทฤษฎี (ตามภาพตัวอย่าง 5.2)
        # -------------------------------------------------------
        
        # 1.1 นโยบายแบบสุ่ม (Randomized Storage)
        denom_random = I + W
        Q_random = np.sqrt((2 * D * P) / denom_random)
        
        # TC ต่อหน่วย (Random) = C + P/Q + (I+W)Q/2D
        holding_cost_unit_random = (denom_random * Q_random) / (2 * D)
        TC_unit_random = C + (P / Q_random) + holding_cost_unit_random

        # 1.2 นโยบายกำหนดพื้นที่ (Dedicated Storage)
        denom_dedicated = I + (2 * W)
        Q_dedicated = np.sqrt((2 * D * P) / denom_dedicated)
        
        # TC ต่อหน่วย (Dedicated) = C + P/Q + I(Q/2)/D + W(Q)/D
        interest_unit_dedi = (I * (Q_dedicated / 2)) / D
        storage_unit_dedi = (W * Q_dedicated) / D
        TC_unit_dedicated = C + (P / Q_dedicated) + interest_unit_dedi + storage_unit_dedi

        cost_analysis_results.append({
            "สินค้า": item['product_name'],
            "Q* (แบบสุ่ม)": Q_random,
            "TC/หน่วย (แบบสุ่ม)": TC_unit_random,
            "Q* (กำหนดพื้นที่)": Q_dedicated,
            "TC/หน่วย (กำหนดพื้นที่)": TC_unit_dedicated,
        })

        # -------------------------------------------------------
        # ส่วนที่ 2: การวางแผนปฏิบัติงานจริง (ตามภาพตัวอย่าง 5.3)
        # *อ้างอิงข้อมูลจาก "นโยบายแบบสุ่ม" มาคำนวณต่อ*
        # -------------------------------------------------------
        
        # ใช้ Q* จากแบบสุ่ม (ปัดเศษเป็นจำนวนเต็มตามหลักปฏิบัติ)
        Q_target = round(Q_random) 
        
        # คำนวณจำนวนครั้งการสั่งซื้อ (N)
        N_orders = D / Q_target
        N_orders_practical = round(N_orders) # ปัดเป็นจำนวนเต็ม (เช่น 51 ครั้ง)
        
        # อัตราการใช้ต่อวัน (d)
        usage_rate_day = D / days_yr
        d_practical = round(usage_rate_day) # ปัดเศษ (เช่น 27 หน่วย/วัน)
        
        # ช่วงเวลาระหว่างการสั่ง (T) = วันทำงาน / จำนวนครั้ง
        if N_orders_practical > 0:
            T_days = days_yr / N_orders_practical
            T_practical = round(T_days) # ปัดเศษ (เช่น 6 วัน)
        else:
            T_practical = 0

        # จุดสั่งซื้อ (Reorder Point: RP)
        RP = d_practical * L #
        
        # ปริมาณสั่งซื้อจริงหน้างาน (Q actual) = อัตราใช้ x รอบวันสั่งซื้อ
        Q_actual = d_practical * T_practical # (เช่น 27*6 = 162 หน่วย)
        
        # คำนวณค่าใช้จ่ายรวมต่อปี (Total Annual Cost) สำหรับแผนจริง
        annual_ordering_cost = N_orders_practical * P
        # ค่าเก็บรักษาคิดจาก Q_actual (เฉลี่ย Q/2) * (I+W)
        annual_holding_cost = (I + W) * (Q_actual / 2)
        total_annual_cost = annual_ordering_cost + annual_holding_cost

        operational_plan_results.append({
            "สินค้า": item['product_name'],
            "Q (ทฤษฎี)": Q_target,
            "จำนวนสั่ง (ครั้ง/ปี)": N_orders_practical,
            "รอบสั่ง (วัน)": T_practical,
            "อัตราใช้ (หน่วย/วัน)": d_practical,
            "จุดสั่งซื้อ (RP)": RP,
            "Q (สั่งจริง)": Q_actual,
            "ค่าสั่งซื้อ (บาท/ปี)": annual_ordering_cost,
            "ค่าเก็บรักษา (บาท/ปี)": annual_holding_cost,
            "รวมค่าใช้จ่าย (บาท/ปี)": total_annual_cost
        })

    return pd.DataFrame(cost_analysis_results), pd.DataFrame(operational_plan_results)

# ==========================================
# 3. แสดงผลลัพธ์
# ==========================================
df_costs, df_plan = analyze_inventory(inventory_data)

# จัดรูปแบบตัวเลขให้สวยงาม
pd.options.display.float_format = '{:,.2f}'.format

print("--- ตารางที่ 1: เปรียบเทียบต้นทุน EOQ (Random vs Dedicated) [ตามภาพ 5.2] ---")
display(df_costs)

print("\n--- ตารางที่ 2: แผนการดำเนินงานจริง (Operational Plan) [ตามภาพ 5.3] ---")
print("*คำนวณโดยใช้นโยบายแบบสุ่ม (Randomized) และมีการปัดเศษเพื่อความสะดวกหน้างาน")
display(df_plan)

--- ตารางที่ 1: เปรียบเทียบต้นทุน EOQ (Random vs Dedicated) [ตามภาพ 5.2] ---


,สินค้า,Q* (แบบสุ่ม),TC/หน่วย (แบบสุ่ม),Q* (กำหนดพื้นที่),TC/หน่วย (กำหนดพื้นที่)
0,สินค้าตัวอย่าง (Ex 5.2 & 5.3),156.72,"1,208.93",129.10,"1,210.84"



--- ตารางที่ 2: แผนการดำเนินงานจริง (Operational Plan) [ตามภาพ 5.3] ---
*คำนวณโดยใช้นโยบายแบบสุ่ม (Randomized) และมีการปัดเศษเพื่อความสะดวกหน้างาน


,สินค้า,Q (ทฤษฎี),จำนวนสั่ง (ครั้ง/ปี),รอบสั่ง (วัน),อัตราใช้ (หน่วย/วัน),จุดสั่งซื้อ (RP),Q (สั่งจริง),ค่าสั่งซื้อ (บาท/ปี),ค่าเก็บรักษา (บาท/ปี),รวมค่าใช้จ่าย (บาท/ปี)
0,สินค้าตัวอย่าง (Ex 5.2 & 5.3),157,51,6,27,54,162,35700,"36,936.00","72,636.00"


In [4]:
import pandas as pd
import numpy as np
import math

# ==========================================
# 1. ส่วนกำหนดข้อมูล (User Input)
# ==========================================
# คุณสามารถ เพิ่ม/ลบ สินค้าใน list นี้ได้ตามต้องการ
# โปรแกรมจะคำนวณทั้ง 3 รูปแบบ (Ex 5.2, 5.3, 5.4) ให้กับทุกสินค้า
inventory_data = [
    {
        "product_name": "สินค้าตัวอย่าง (Ex 5.2-5.4)",
        "D": 8000,        # ความต้องการ (หน่วย/ปี)
        "P": 700,         # ค่าใช้จ่ายในการสั่งซื้อ (บาท/ครั้ง)
        "C": 1200,        # ต้นทุนผลิตภัณฑ์ (บาท/หน่วย)
        "i_percent": 20,  # ดอกเบี้ย (% ต่อปี) - สำหรับ Ex 5.2
        "W_monthly": 18,  # ค่าเช่าพื้นที่เก็บ (บาท/หน่วย/เดือน) - สำหรับ Ex 5.2
        "L": 2,           # Lead Time (วัน) - สำหรับ Ex 5.3
        "working_days": 300, # จำนวนวันทำงานต่อปี - สำหรับ Ex 5.3
        "H_percent": 30   # ค่าเก็บรักษารวม (% ของต้นทุน) - สำหรับ Ex 5.4
    },
    # ลองเพิ่มสินค้าใหม่ได้ที่นี่
    # {
    #     "product_name": "สินค้าทดสอบ B",
    #     "D": 5000, "P": 500, "C": 800, "i_percent": 15, "W_monthly": 10,
    #     "L": 5, "working_days": 365, "H_percent": 25
    # }
]

# ==========================================
# 2. ฟังก์ชันคำนวณ
# ==========================================
def analyze_full_inventory(data_list):
    results_5_2 = [] # ผลลัพธ์ Ex 5.2 (Random vs Dedicated)
    results_5_3 = [] # ผลลัพธ์ Ex 5.3 (Operational Plan)
    results_5_4 = [] # ผลลัพธ์ Ex 5.4 (Simplified H)

    for item in data_list:
        # --- ดึงตัวแปร ---
        D, P, C = item['D'], item['P'], item['C']
        i, W_mo = item['i_percent'] / 100, item['W_monthly']
        L, days_yr = item['L'], item['working_days']
        H_pct = item['H_percent'] / 100

        # ค่าคงที่เบื้องต้น
        I = i * C              # Interest Charge (บาท/หน่วย/ปี)
        W = W_mo * 12          # Storage Charge (บาท/หน่วย/ปี)

        # -------------------------------------------------------
        # Part 1: Ex 5.2 (Random vs Dedicated)
        # -------------------------------------------------------
        # 1.1 แบบสุ่ม (Randomized)
        denom_random = I + W
        Q_random = np.sqrt((2 * D * P) / denom_random)
        
        # TC (Random) = C + P/Q + (I+W)Q/2D
        holding_cost_unit_random = (denom_random * Q_random) / (2 * D)
        TC_unit_random = C + (P / Q_random) + holding_cost_unit_random

        # 1.2 แบบกำหนดพื้นที่ (Dedicated)
        denom_dedicated = I + (2 * W)
        Q_dedicated = np.sqrt((2 * D * P) / denom_dedicated)
        
        # TC (Dedicated) = C + P/Q + I(Q/2)/D + W(Q)/D
        interest_unit_dedi = (I * (Q_dedicated / 2)) / D
        storage_unit_dedi = (W * Q_dedicated) / D
        TC_unit_dedicated = C + (P / Q_dedicated) + interest_unit_dedi + storage_unit_dedi

        results_5_2.append({
            "สินค้า": item['product_name'],
            "Q* (สุ่ม)": Q_random,
            "TC/หน่วย (สุ่ม)": TC_unit_random,
            "Q* (กำหนดที่)": Q_dedicated,
            "TC/หน่วย (กำหนดที่)": TC_unit_dedicated
        })

        # -------------------------------------------------------
        # Part 2: Ex 5.3 (Operational Plan - แผนหน้างานจริง)
        # *อ้างอิง Q จากแบบสุ่ม*
        # -------------------------------------------------------
        Q_target = round(Q_random)  # ปัดเป็นจำนวนเต็ม
        
        # 1. จำนวนครั้ง (N)
        N_exact = D / Q_target
        N_practical = round(N_exact) # ปัดตามหลักปฏิบัติ
        
        # 2. รอบเวลา (T)
        if N_practical > 0:
            T_practical = round(days_yr / N_practical) # ปัดเศษวัน
        else:
            T_practical = 0

        # 3. อัตราใช้ต่อวัน (d)
        d_practical = round(D / days_yr) #

        # 4. จุดสั่งซื้อ (RP) และ ปริมาณสั่งจริง (Q actual)
        RP = d_practical * L
        Q_actual = d_practical * T_practical # (d x T)
        
        # 5. คำนวณค่าใช้จ่ายรวมต่อปี (Total Annual Cost)
        annual_order_cost = N_practical * P
        annual_hold_cost = (I + W) * (Q_actual / 2) # คิดจาก Q actual
        total_annual_cost = annual_order_cost + annual_hold_cost

        results_5_3.append({
            "สินค้า": item['product_name'],
            "N (ครั้ง/ปี)": N_practical,
            "T (วัน)": T_practical,
            "d (หน่วย/วัน)": d_practical,
            "RP (จุดสั่งซื้อ)": RP,
            "Q (สั่งจริง)": Q_actual,
            "รวมค่าใช้จ่าย (บาท/ปี)": total_annual_cost
        })

        # -------------------------------------------------------
        # Part 3: Ex 5.4 (Simplified H - กรณีประเมินแยกยาก)
        # -------------------------------------------------------
        H_val = H_pct * C # คำนวณ H เป็นบาท
        
        # สูตร Q = sqrt(2DP / H)
        Q_simple = np.sqrt((2 * D * P) / H_val)

        results_5_4.append({
            "สินค้า": item['product_name'],
            "H (% ของทุน)": item['H_percent'],
            "H (บาท/หน่วย/ปี)": H_val,
            "Q* (แบบย่อ)": Q_simple
        })

    return pd.DataFrame(results_5_2), pd.DataFrame(results_5_3), pd.DataFrame(results_5_4)

# ==========================================
# 3. แสดงผลลัพธ์
# ==========================================
df52, df53, df54 = analyze_full_inventory(inventory_data)
pd.options.display.float_format = '{:,.2f}'.format

print("--- 1. เปรียบเทียบต้นทุน EOQ (Ex 5.2) ---")
display(df52)

print("\n--- 2. แผนการดำเนินงานจริง (Ex 5.3) ---")
print("*คำนวณโดยใช้ Q แบบสุ่ม และมีการปัดเศษเพื่อการใช้งานจริง")
display(df53)

print("\n--- 3. การคำนวณแบบประมาณการ H (Ex 5.4) ---")
print("*ใช้กรณีแยกค่า I และ W ไม่ได้")
display(df54)

--- 1. เปรียบเทียบต้นทุน EOQ (Ex 5.2) ---


,สินค้า,Q* (สุ่ม),TC/หน่วย (สุ่ม),Q* (กำหนดที่),TC/หน่วย (กำหนดที่)
0,สินค้าตัวอย่าง (Ex 5.2-5.4),156.72,"1,208.93",129.10,"1,210.84"



--- 2. แผนการดำเนินงานจริง (Ex 5.3) ---
*คำนวณโดยใช้ Q แบบสุ่ม และมีการปัดเศษเพื่อการใช้งานจริง


,สินค้า,N (ครั้ง/ปี),T (วัน),d (หน่วย/วัน),RP (จุดสั่งซื้อ),Q (สั่งจริง),รวมค่าใช้จ่าย (บาท/ปี)
0,สินค้าตัวอย่าง (Ex 5.2-5.4),51,6,27,54,162,"72,636.00"



--- 3. การคำนวณแบบประมาณการ H (Ex 5.4) ---
*ใช้กรณีแยกค่า I และ W ไม่ได้


,สินค้า,H (% ของทุน),H (บาท/หน่วย/ปี),Q* (แบบย่อ)
0,สินค้าตัวอย่าง (Ex 5.2-5.4),30,360.00,176.38


In [5]:
import pandas as pd
import numpy as np
import math

# ==========================================
# 1. ส่วนกำหนดข้อมูล (User Input)
# ==========================================
production_data = [
    # กรณีที่ 1: ข้อมูลจากตัวอย่างที่ 5.5 (ให้ค่า Holding รวมมาเป็น %)
    {
        "case_name": "ตัวอย่างที่ 5.5 (เฟอร์นิเจอร์)",
        "D": 10200,       # ความต้องการ (หน่วย/ปี)
        "P": 5000,        # ค่าใช้จ่ายในการตั้งเครื่องจักร (บาท/ครั้ง)
        "C": 1200,        # ต้นทุนผลิตภัณฑ์ (บาท/หน่วย)
        "days_yr": 288,   # วันทำงานต่อปี
        "M_daily": 140,   # อัตราการผลิต (หน่วย/วัน)
        "H_percent": 30,  # ค่าเก็บรักษารวม (% ของต้นทุน) - ใช้สำหรับ Random Storage ทั่วไป
        "I_percent": None,# (Optional) ดอกเบี้ย - ถ้าใส่จะคำนวณ Dedicated ได้
        "W_unit": None    # (Optional) ค่าเช่าที่ (บาท/หน่วย/ปี) - ถ้าใส่จะคำนวณ Dedicated ได้
    },
    # กรณีที่ 2: ลองสมมติถ้าแยก I และ W ได้ (เพื่อทดสอบสูตร 5.20 ในภาพทฤษฎี)
    {
        "case_name": "ทดสอบสูตร Dedicated (สมมติ)",
        "D": 10200,
        "P": 5000,
        "C": 1200,
        "days_yr": 288,
        "M_daily": 140,
        "H_percent": None, 
        "I_percent": 20,   # สมมติ I = 20%
        "W_unit": 120      # สมมติ W = 120 บาท/หน่วย/ปี (10% ของ 1200)
    }
]

# ==========================================
# 2. ฟังก์ชันคำนวณ EMQ (Production Model)
# ==========================================
def analyze_production_model(data_list):
    results = []

    for item in data_list:
        # --- ดึงตัวแปร ---
        D = item['D']
        P = item['P']
        C = item['C']
        days = item['days_yr']
        M_daily = item['M_daily']
        
        # คำนวณตัวแปรพื้นฐาน
        M_annual = M_daily * days       # อัตราผลิตต่อปี
        d_daily = D / days              # อัตราใช้ต่อวัน
        
        # Factor การผลิต (1 - D/M) หรือ (1 - d/p)
        # หมายเหตุ: ในหนังสือใช้ d=35 (ปัดเศษ) เพื่อให้ได้ 1-35/140 = 0.75
        # โค้ดนี้จะใช้ค่าละเอียดเพื่อความแม่นยำ แต่ผลลัพธ์จะใกล้เคียงกันมาก
        production_factor = 1 - (D / M_annual)

        # เตรียมค่าใช้จ่ายในการเก็บรักษา (H)
        # กรณีระบุ H รวม (Ex 5.5)
        if item['H_percent'] is not None:
            H_val = (item['H_percent'] / 100) * C
            I_val = 0 # ไม่ได้แยก
            W_val = 0 # ไม่ได้แยก
            calc_dedicated = False # คำนวณ Dedicated ไม่ได้ถ้าไม่แยก I, W
            
            # ใช้ H แทน (I+W) ในสูตร Random
            denom_random = H_val * production_factor
            
        # กรณีระบุแยก I และ W (ตามทฤษฎี Ex 5.20)
        else:
            I_val = (item['I_percent'] / 100) * C
            W_val = item['W_unit']
            calc_dedicated = True
            
            # คำนวณตัวหารสำหรับ Random (I+W)
            denom_random = (I_val + W_val) * production_factor

        # -------------------------------------------------------
        # 2.1 คำนวณแบบ Random Storage (สมการ 5.19)
        # -------------------------------------------------------
        Q_opt_random = np.sqrt((2 * D * P) / denom_random)
        
        # ปริมาณคงคลังสูงสุด (Q_max)
        Q_max_random = Q_opt_random * production_factor
        
        # คำนวณ TC (Random)
        # TC = C(D) + (P)(D/Q) + (H)(Q_max/2) -- H ในที่นี้คือ I+W
        holding_cost_random = denom_random * Q_opt_random / 2 # ยบย่อสูตรจาก (I+W)(1-D/M)(Q)/2
        TC_random = (C * D) + (P * D / Q_opt_random) + holding_cost_random

        # -------------------------------------------------------
        # 2.2 คำนวณแบบ Dedicated Storage (สมการ 5.20)
        # -------------------------------------------------------
        if calc_dedicated:
            # ตัวหารสำหรับ Dedicated คือ (I + 2W)(1 - D/M)
            denom_dedicated = (I_val + (2 * W_val)) * production_factor
            
            Q_opt_dedicated = np.sqrt((2 * D * P) / denom_dedicated)
            Q_max_dedicated = Q_opt_dedicated * production_factor
            
            # TC (Dedicated)
            # TC = C(D) + P(D/Q) + I(Q_max/2) + W(Q_max)
            # สูตรย่อ: TC = ... + (I+2W)(Q_max)/2
            holding_cost_dedicated = denom_dedicated * Q_opt_dedicated / 2
            TC_dedicated = (C * D) + (P * D / Q_opt_dedicated) + holding_cost_dedicated
        else:
            Q_opt_dedicated = None
            TC_dedicated = None
            Q_max_dedicated = None

        # บันทึกผลลัพธ์
        results.append({
            "กรณีศึกษา": item['case_name'],
            "อัตราผลิต (M)": M_annual,
            "อัตราใช้ (D)": D,
            "ตัวประกอบ (1-D/M)": production_factor,
            "H หรือ (I+W)": H_val if item['H_percent'] else (I_val + W_val),
            # ผลลัพธ์ Random
            "Q* (Random)": Q_opt_random,
            "Q_max (พื้นที่เก็บ)": Q_max_random,
            "TC รวม (บาท/ปี)": TC_random,
            # ผลลัพธ์ Dedicated
            "Q* (Dedicated)": Q_opt_dedicated,
            "TC Dedicated": TC_dedicated
        })

    return pd.DataFrame(results)

# ==========================================
# 3. แสดงผลลัพธ์
# ==========================================
df_production = analyze_production_model(production_data)

# จัดรูปแบบตัวเลข
pd.options.display.float_format = '{:,.2f}'.format

print("--- ผลการคำนวณปริมาณการสั่งผลิตที่ประหยัด (EMQ) ---")
print("อ้างอิงสูตร 5.19 (Random) และ 5.20 (Dedicated) จากภาพประกอบ")
display(df_production.T) # .T เพื่อให้อ่านง่ายในแนวตั้ง

--- ผลการคำนวณปริมาณการสั่งผลิตที่ประหยัด (EMQ) ---
อ้างอิงสูตร 5.19 (Random) และ 5.20 (Dedicated) จากภาพประกอบ


,0,1
กรณีศึกษา,ตัวอย่างที่ 5.5 (เฟอร์นิเจอร์),ทดสอบสูตร Dedicated (สมมติ)
อัตราผลิต (M),40320,40320
อัตราใช้ (D),10200,10200
ตัวประกอบ (1-D/M),0.75,0.75
H หรือ (I+W),360.00,360.00
Q* (Random),615.86,615.86
Q_max (พื้นที่เก็บ),460.06,460.06
TC รวม (บาท/ปี),"12,405,622.20","12,405,622.20"
Q* (Dedicated),NaN,533.35
TC Dedicated,NaN,"12,431,244.05"


In [6]:
import pandas as pd
import numpy as np

# ==========================================
# 1. ส่วนกำหนดข้อมูล (User Input)
# ==========================================
discount_cases = [
    # --- กรณีที่ 1: ตัวอย่างที่ 5.6 (ค่าเก็บรักษาคงที่) ---
    {
        "case_name": "Ex 5.6 (Fixed Holding Cost)",
        "D": 5500,        # ความต้องการ (หน่วย/เดือน)
        "P": 15000,       # ค่าสั่งซื้อ (บาท/ครั้ง)
        "holding_type": "fixed", # ประเภทค่าเก็บรักษา
        "H_value": 63,    # ค่าเก็บรักษา (บาท/หน่วย/เดือน) - ระบุค่าตรงๆ
        "time_factor": 1, # ตัวหารปรับหน่วยเวลา (1 เพราะหน่วย H ตรงกับ D แล้ว)
        "price_schedule": [
            {"min": 1,    "max": 1499, "price": 3000},
            {"min": 1500, "max": 1999, "price": 2850},
            {"min": 2000, "max": float('inf'), "price": 2700}
        ]
    },
    # --- กรณีที่ 2: ตัวอย่างที่ 5.7 (ค่าเก็บรักษาเป็น % รายปี) ---
    {
        "case_name": "Ex 5.7 (% Holding Cost Annual)",
        "D": 5500,        # ความต้องการ (หน่วย/เดือน)
        "P": 15000,       # ค่าสั่งซื้อ (บาท/ครั้ง)
        "holding_type": "percent", 
        "H_value": 25,    # ค่าเก็บรักษา (25% ต่อปี)
        "time_factor": 12, # ต้องหาร 12 เพื่อแปลง % ปี เป็น เดือน (ให้ตรงกับหน่วย D)
        "price_schedule": [
            {"min": 1,    "max": 1499, "price": 3000},
            {"min": 1500, "max": 1999, "price": 2850},
            {"min": 2000, "max": float('inf'), "price": 2700}
        ]
    }
]

# ==========================================
# 2. ฟังก์ชันคำนวณ Quantity Discount
# ==========================================
def analyze_quantity_discount(cases):
    final_summary = []
    
    for case in cases:
        print(f"\n{'='*50}")
        print(f"กำลังวิเคราะห์: {case['case_name']}")
        print(f"{'='*50}")
        
        D = case['D']
        P = case['P']
        factor = case['time_factor']
        schedule = case['price_schedule']
        
        candidates = []
        
        # วนลูปคำนวณทุกช่วงราคา
        for tier in schedule:
            price = tier['price']
            min_q = tier['min']
            max_q = tier['max']
            
            # 1. คำนวณ H (ต้นทุนการเก็บรักษาต่อหน่วยต่อเวลา)
            if case['holding_type'] == 'fixed':
                H = case['H_value'] # ใช้ค่าที่ระบุเลย (เช่น 63 บาท)
            else:
                # สูตร: H = (i% * Price) / time_factor
                # ตัวอย่าง: (0.25 * 2700) / 12
                H = (case['H_value'] / 100 * price) / factor
            
            # 2. คำนวณ EOQ ตามสูตร 5.9 หรือ 5.15
            # Q = sqrt(2DP / H)
            Q_calc = np.sqrt((2 * D * P) / H)
            
            # 3. ตรวจสอบเงื่อนไขช่วงปริมาณ (Feasibility Check)
            if min_q <= Q_calc <= max_q:
                Q_valid = Q_calc
                status = "Valid EOQ (อยู่ในช่วง)"
                is_candidate = True
            elif Q_calc < min_q:
                Q_valid = min_q # ปรับขึ้นไปที่จุดต่ำสุดของช่วง (Price Break)
                status = "Adjusted to Min (ปรับเป็นขั้นต่ำ)"
                is_candidate = True
            else: # Q_calc > max_q
                Q_valid = Q_calc
                status = "Exceeds Range (เกินช่วง - มักไม่เลือก)"
                # โดยทั่วไปถ้าราคาถูกกว่าแต่ Q* ดันไปตกในช่วงราคาแพงกว่า จะไม่เป็นจริง
                # แต่โค้ดจะคำนวณไว้ให้ดูเปรียบเทียบ
                is_candidate = False 

            if is_candidate:
                # 4. คำนวณ Total Cost (TC)
                # TC Total = (Price * D) + (Ordering Cost) + (Holding Cost)
                # หมายเหตุ: ในหนังสือใช้ D * TC_unit ซึ่งความหมายเดียวกันคือ Total Annual/Monthly Cost
                product_cost = price * D
                ordering_cost = (P * D) / Q_valid
                holding_cost = (H * Q_valid) / 2
                total_cost = product_cost + ordering_cost + holding_cost
                
                candidates.append({
                    "Price": price,
                    "Range": f"{min_q:,.0f} - {max_q}",
                    "H (บาท/หน่วย)": H,
                    "Q (คำนวณ)": Q_calc,
                    "Q (ที่เลือกใช้)": Q_valid,
                    "สถานะ": status,
                    "ค่าสินค้า": product_cost,
                    "ค่าสั่งซื้อ": ordering_cost,
                    "ค่าเก็บรักษา": holding_cost,
                    "รวมค่าใช้จ่าย (Total Cost)": total_cost
                })

        # สร้างตารางเปรียบเทียบของเคสนี้
        df_results = pd.DataFrame(candidates)
        
        # หาค่าต่ำสุด
        best_row = df_results.loc[df_results['รวมค่าใช้จ่าย (Total Cost)'].idxmin()]
        
        # แสดงผล
        pd.options.display.float_format = '{:,.2f}'.format
        display(df_results)
        print(f">> สรุป {case['case_name']}: ควรเลือกสั่ง {best_row['Q (ที่เลือกใช้)']:,.2f} หน่วย (ที่ราคา {best_row['Price']})")
        print(f">> ค่าใช้จ่ายรวมต่ำสุด: {best_row['รวมค่าใช้จ่าย (Total Cost)']:,.2f} บาท")

# ==========================================
# 3. เรียกใช้งาน
# ==========================================
analyze_quantity_discount(discount_cases)


กำลังวิเคราะห์: Ex 5.6 (Fixed Holding Cost)


,Price,Range,H (บาท/หน่วย),Q (คำนวณ),Q (ที่เลือกใช้),สถานะ,ค่าสินค้า,ค่าสั่งซื้อ,ค่าเก็บรักษา,รวมค่าใช้จ่าย (Total Cost)
0,2850,"1,500 - 1999",63,"1,618.35","1,618.35",Valid EOQ (อยู่ในช่วง),15675000,"50,977.94","50,977.94","15,776,955.87"
1,2700,"2,000 - inf",63,"1,618.35","2,000.00",Adjusted to Min (ปรับเป็นขั้นต่ำ),14850000,"41,250.00","63,000.00","14,954,250.00"


>> สรุป Ex 5.6 (Fixed Holding Cost): ควรเลือกสั่ง 2,000.00 หน่วย (ที่ราคา 2700)
>> ค่าใช้จ่ายรวมต่ำสุด: 14,954,250.00 บาท

กำลังวิเคราะห์: Ex 5.7 (% Holding Cost Annual)


,Price,Range,H (บาท/หน่วย),Q (คำนวณ),Q (ที่เลือกใช้),สถานะ,ค่าสินค้า,ค่าสั่งซื้อ,ค่าเก็บรักษา,รวมค่าใช้จ่าย (Total Cost)
0,2850,"1,500 - 1999",59.38,"1,667.02","1,667.02",Valid EOQ (อยู่ในช่วง),15675000,"49,489.58","49,489.58","15,773,979.16"
1,2700,"2,000 - inf",56.25,"1,712.70","2,000.00",Adjusted to Min (ปรับเป็นขั้นต่ำ),14850000,"41,250.00","56,250.00","14,947,500.00"


>> สรุป Ex 5.7 (% Holding Cost Annual): ควรเลือกสั่ง 2,000.00 หน่วย (ที่ราคา 2700)
>> ค่าใช้จ่ายรวมต่ำสุด: 14,947,500.00 บาท


# ตย.

In [4]:
import pandas as pd
import numpy as np
import math

# ==========================================
# 1. ส่วนกำหนดข้อมูล (User Input)
# ==========================================
production_data = [
    # กรณีที่ 1: ข้อมูลจากตัวอย่างที่ 5.5 (ให้ค่า Holding รวมมาเป็น %)
    {
        "case_name": "ตัวอย่างที่ 5.5 (เฟอร์นิเจอร์)",
        "D": 4000,       # ความต้องการ (หน่วย/ปี)
        "P": 6000,        # ค่าใช้จ่ายในการตั้งเครื่องจักร (บาท/ครั้ง)
        "C": 100,        # ต้นทุนผลิตภัณฑ์ (บาท/หน่วย)
        "days_yr": 365,   # วันทำงานต่อปี
        "M_daily": 140,   # อัตราการผลิต (หน่วย/วัน)
        "H_percent": 30,  # ค่าเก็บรักษารวม (% ของต้นทุน) - ใช้สำหรับ Random Storage ทั่วไป
        "I_percent": None,# (Optional) ดอกเบี้ย - ถ้าใส่จะคำนวณ Dedicated ได้
        "W_unit": None    # (Optional) ค่าเช่าที่ (บาท/หน่วย/ปี) - ถ้าใส่จะคำนวณ Dedicated ได้
    },
    # กรณีที่ 2: ลองสมมติถ้าแยก I และ W ได้ (เพื่อทดสอบสูตร 5.20 ในภาพทฤษฎี)
    {
        "case_name": "ทดสอบสูตร Dedicated (สมมติ)",
        "D": 10200,
        "P": 5000,
        "C": 1200,
        "days_yr": 288,
        "M_daily": 140,
        "H_percent": None, 
        "I_percent": 20,   # สมมติ I = 20%
        "W_unit": 120      # สมมติ W = 120 บาท/หน่วย/ปี (10% ของ 1200)
    }
]

# ==========================================
# 2. ฟังก์ชันคำนวณ EMQ (Production Model)
# ==========================================
def analyze_production_model(data_list):
    results = []

    for item in data_list:
        # --- ดึงตัวแปร ---
        D = item['D']
        P = item['P']
        C = item['C']
        days = item['days_yr']
        M_daily = item['M_daily']
        
        # คำนวณตัวแปรพื้นฐาน
        M_annual = M_daily * days       # อัตราผลิตต่อปี
        d_daily = D / days              # อัตราใช้ต่อวัน
        
        # Factor การผลิต (1 - D/M) หรือ (1 - d/p)
        # หมายเหตุ: ในหนังสือใช้ d=35 (ปัดเศษ) เพื่อให้ได้ 1-35/140 = 0.75
        # โค้ดนี้จะใช้ค่าละเอียดเพื่อความแม่นยำ แต่ผลลัพธ์จะใกล้เคียงกันมาก
        production_factor = 1 - (D / M_annual)

        # เตรียมค่าใช้จ่ายในการเก็บรักษา (H)
        # กรณีระบุ H รวม (Ex 5.5)
        if item['H_percent'] is not None:
            H_val = (item['H_percent'] / 100) * C
            I_val = 0 # ไม่ได้แยก
            W_val = 0 # ไม่ได้แยก
            calc_dedicated = False # คำนวณ Dedicated ไม่ได้ถ้าไม่แยก I, W
            
            # ใช้ H แทน (I+W) ในสูตร Random
            denom_random = H_val * production_factor
            
        # กรณีระบุแยก I และ W (ตามทฤษฎี Ex 5.20)
        else:
            I_val = (item['I_percent'] / 100) * C
            W_val = item['W_unit']
            calc_dedicated = True
            
            # คำนวณตัวหารสำหรับ Random (I+W)
            denom_random = (I_val + W_val) * production_factor

        # -------------------------------------------------------
        # 2.1 คำนวณแบบ Random Storage (สมการ 5.19)
        # -------------------------------------------------------
        Q_opt_random = np.sqrt((2 * D * P) / denom_random)
        
        # ปริมาณคงคลังสูงสุด (Q_max)
        Q_max_random = Q_opt_random * production_factor
        
        # คำนวณ TC (Random)
        # TC = C(D) + (P)(D/Q) + (H)(Q_max/2) -- H ในที่นี้คือ I+W
        holding_cost_random = denom_random * Q_opt_random / 2 # ยบย่อสูตรจาก (I+W)(1-D/M)(Q)/2
        TC_random = (C * D) + (P * D / Q_opt_random) + holding_cost_random

        # -------------------------------------------------------
        # 2.2 คำนวณแบบ Dedicated Storage (สมการ 5.20)
        # -------------------------------------------------------
        if calc_dedicated:
            # ตัวหารสำหรับ Dedicated คือ (I + 2W)(1 - D/M)
            denom_dedicated = (I_val + (2 * W_val)) * production_factor
            
            Q_opt_dedicated = np.sqrt((2 * D * P) / denom_dedicated)
            Q_max_dedicated = Q_opt_dedicated * production_factor
            
            # TC (Dedicated)
            # TC = C(D) + P(D/Q) + I(Q_max/2) + W(Q_max)
            # สูตรย่อ: TC = ... + (I+2W)(Q_max)/2
            holding_cost_dedicated = denom_dedicated * Q_opt_dedicated / 2
            TC_dedicated = (C * D) + (P * D / Q_opt_dedicated) + holding_cost_dedicated
        else:
            Q_opt_dedicated = None
            TC_dedicated = None
            Q_max_dedicated = None

        # บันทึกผลลัพธ์
        results.append({
            "กรณีศึกษา": item['case_name'],
            "อัตราผลิต (M)": M_annual,
            "อัตราใช้ (D)": D,
            "ตัวประกอบ (1-D/M)": production_factor,
            "H หรือ (I+W)": H_val if item['H_percent'] else (I_val + W_val),
            # ผลลัพธ์ Random
            "Q* (Random)": Q_opt_random,
            "Q_max (พื้นที่เก็บ)": Q_max_random,
            "TC รวม (บาท/ปี)": TC_random,
            # ผลลัพธ์ Dedicated
            "Q* (Dedicated)": Q_opt_dedicated,
            "TC Dedicated": TC_dedicated
        })

    return pd.DataFrame(results)

# ==========================================
# 3. แสดงผลลัพธ์
# ==========================================
df_production = analyze_production_model(production_data)

# จัดรูปแบบตัวเลข
pd.options.display.float_format = '{:,.2f}'.format

print("--- ผลการคำนวณปริมาณการสั่งผลิตที่ประหยัด (EMQ) ---")
print("อ้างอิงสูตร 5.19 (Random) และ 5.20 (Dedicated) จากภาพประกอบ")
display(df_production.T) # .T เพื่อให้อ่านง่ายในแนวตั้ง

--- ผลการคำนวณปริมาณการสั่งผลิตที่ประหยัด (EMQ) ---
อ้างอิงสูตร 5.19 (Random) และ 5.20 (Dedicated) จากภาพประกอบ


,0,1
กรณีศึกษา,ตัวอย่างที่ 5.5 (เฟอร์นิเจอร์),ทดสอบสูตร Dedicated (สมมติ)
อัตราผลิต (M),51100,40320
อัตราใช้ (D),4000,10200
ตัวประกอบ (1-D/M),0.92,0.75
H หรือ (I+W),30.00,360.00
Q* (Random),"1,317.53",615.86
Q_max (พื้นที่เก็บ),"1,214.40",460.06
TC รวม (บาท/ปี),"436,431.85","12,405,622.20"
Q* (Dedicated),NaN,533.35
TC Dedicated,NaN,"12,431,244.05"
